In [32]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [33]:
!pip install jenkspy

In [34]:
!pip install scikit-posthocs

In [35]:
df = pd.read_csv("data_cleaning_2026_27_04.csv")

In [36]:
df.columns

Index(['id', 'motivo_ausencia', 'mes_ausencia', 'dia_semana', 'estaciones',
       'gasto_transporte', 'distancia_casa_trabajo', 'tiempo_servicio', 'edad',
       'carga_trabajo_diaria', 'cumplimiento_objetivo', 'falta_disciplinaria',
       'educacion', 'hijos', 'bebedor_social', 'fumador_social', 'mascota',
       'peso', 'estatura', 'indice_masa_corporal', 'horas_ausentismo',
       'indicador_enfermedad', 'motivo_ausencia_mapeado',
       'mes_ausencia_mapeado', 'educacion_mapeada'],
      dtype='object')

In [37]:
df_clean = df.loc[:, ~df.columns.isin(['mascota', 'peso', 'estatura', 'bebedor_social', 'fumador_social'])]

In [38]:
df_clean = df_clean[df_clean['horas_ausentismo'] > 0]
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 757 entries, 0 to 805
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       757 non-null    int64  
 1   motivo_ausencia          757 non-null    int64  
 2   mes_ausencia             757 non-null    float64
 3   dia_semana               757 non-null    object 
 4   estaciones               757 non-null    object 
 5   gasto_transporte         757 non-null    int64  
 6   distancia_casa_trabajo   757 non-null    int64  
 7   tiempo_servicio          757 non-null    int64  
 8   edad                     757 non-null    int64  
 9   carga_trabajo_diaria     757 non-null    float64
 10  cumplimiento_objetivo    757 non-null    int64  
 11  falta_disciplinaria      757 non-null    int64  
 12  educacion                757 non-null    int64  
 13  hijos                    757 non-null    int64  
 14  indice_masa_corporal     757 no

## Tener hijos incide sul tiempo de ausencia?   
Tests no paramétricos
La variable horas_ausentismo presenta asimetría positiva (muchas ausencias cortas, pocas muy largas), por lo que no sigue distribución normal. Los tests paramétricos clásicos (t-test, ANOVA, Pearson) asumen normalidad y son sensibles a outliers — en este dataset darían resultados poco fiables. Se sustituyeron por sus equivalentes no paramétricos: Mann-Whitney U para comparar dos grupos.

In [39]:
from scipy import stats

con_hijos = df[df['hijos'] > 0]['horas_ausentismo']
sin_hijos = df[df['hijos'] == 0]['horas_ausentismo']

stat, p = stats.mannwhitneyu(con_hijos, sin_hijos, alternative='two-sided')
print(f"p-valor: {p:.4f}")
print("Diferencia significativa" if p < 0.05 else "Sin diferencia significativa")

print(f"\nMediana con hijos:  {con_hijos.median():.1f} h")
print(f"Mediana sin hijos:  {sin_hijos.median():.1f} h")

p-valor: 0.1380
Sin diferencia significativa

Mediana con hijos:  4.0 h
Mediana sin hijos:  3.0 h


## ¿El rango de edad influye en la duración de los episodios de ausencia?   
El rango de edad: el metodo para encontrar los cortes donde la variación dentro del grupo es mínima y entre grupos es máxima. Diseñado para agrupar una variable continua de forma objetiva: agrupación de edad con Jenks Natural Breaks  
Para aplicar Kruskal-Wallis, edad necesita convertirse en grupos discretos. Definir los cortes manualmente (ej. 18-30, 31-45...) introduce sesgo del analista: los resultados del test dependerían de una decisión arbitraria, no de la estructura real de los datos. Se evaluaron tres alternativas: rangos manuales (descartado por sesgo), KMeans (descartado porque está diseñado para espacios multidimensionales y es innecesariamente complejo para una sola variable), y Jenks Natural Breaks. Este último algoritmo encuentra los cortes donde la variación dentro de cada grupo es mínima y la variación entre grupos es máxima — es decir, los grupos resultantes son internamente homogéneos y externamente distintos. Es el método estándar en análisis de una variable continua cuando no se tiene conocimiento previo del dominio para justificar cortes específicos

In [40]:
import jenkspy
import scikit_posthocs as sp
from scipy import stats

# 1. Crear grupos de edad con Jenks Natural Breaks
breaks = jenkspy.jenks_breaks(df_clean['edad'], n_classes=4)
df_clean['rango_edad'] = pd.cut(df_clean['edad'], bins=breaks, include_lowest=True)

print("Distribución de grupos:")
print(df_clean['rango_edad'].value_counts().sort_index())

# 2. Kruskal-Wallis
grupos = [g['horas_ausentismo'].values
          for _, g in df_clean.groupby('rango_edad', observed=True)]

stat, p = stats.kruskal(*grupos)
print(f"\nKruskal-Wallis — p-valor: {p:.4f}")

# 3. Post-hoc Dunn solo si hay diferencia significativa
if p < 0.05:
    print("\nDiferencia significativa — test de Dunn (Bonferroni):")
    print(sp.posthoc_dunn(df_clean, val_col='horas_ausentismo',
                          group_col='rango_edad', p_adjust='bonferroni'))
else:
    print("Sin diferencia significativa entre grupos de edad.")

Distribución de grupos:
rango_edad
(26.999, 31.0]    212
(31.0, 36.0]      149
(36.0, 43.0]      309
(43.0, 58.0]       87
Name: count, dtype: int64

Kruskal-Wallis — p-valor: 0.0000

Diferencia significativa — test de Dunn (Bonferroni):
                (26.999, 31.0]  (31.0, 36.0]  (36.0, 43.0]  (43.0, 58.0]
(26.999, 31.0]    1.000000e+00  2.196447e-07  1.000000e+00  1.000000e+00
(31.0, 36.0]      2.196447e-07  1.000000e+00  2.116258e-07  3.044114e-07
(36.0, 43.0]      1.000000e+00  2.116258e-07  1.000000e+00  7.606020e-01
(43.0, 58.0]      1.000000e+00  3.044114e-07  7.606020e-01  1.000000e+00
